In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))

import fitz          # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import tiktoken
from langchain.text_splitter import RecursiveCharacterTextSplitter

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120

# Token counter (GPT-4 uses cl100k_base)
enc = tiktoken.get_encoding('cl100k_base')

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print('Setup complete ✓')

ModuleNotFoundError: No module named 'fitz'

In [ ]:
PDF_PATH = '../backend/storage/uploads/sample.pdf'   # replace with your PDF

def load_pdf_text(path: str) -> list[dict]:
    """Extracts text page-by-page using PyMuPDF."""
    pages = []
    doc   = fitz.open(path)
    for i, page in enumerate(doc):
        text = page.get_text('text').strip()
        if text:
            pages.append({'page': i + 1, 'text': text})
    doc.close()
    return pages

pages     = load_pdf_text(PDF_PATH)
full_text = '\n\n'.join(p['text'] for p in pages)

print(f'Pages loaded   : {len(pages)}')
print(f'Total chars    : {len(full_text):,}')
print(f'Total tokens   : {count_tokens(full_text):,}')
print(f'\n--- First 400 chars ---\n{full_text[:400]}')

In [ ]:
CHUNK_SIZES = [256, 512, 768, 1024]
OVERLAPS    = [0, 50, 100, 150]

results = []

for size in CHUNK_SIZES:
    for overlap in OVERLAPS:
        if overlap >= size:
            continue

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=overlap,
            separators=['\n\n', '\n', '.', ' ', '']
        )
        chunks = splitter.split_text(full_text)
        token_counts = [count_tokens(c) for c in chunks]

        results.append({
            'chunk_size'  : size,
            'overlap'     : overlap,
            'num_chunks'  : len(chunks),
            'avg_chars'   : round(sum(len(c) for c in chunks) / len(chunks), 1),
            'avg_tokens'  : round(sum(token_counts) / len(token_counts), 1),
            'max_tokens'  : max(token_counts),
            'min_tokens'  : min(token_counts),
        })

df = pd.DataFrame(results)
df.style.background_gradient(subset=['num_chunks', 'avg_tokens'], cmap='YlOrRd')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Number of chunks per config
for overlap in OVERLAPS:
    sub = df[df['overlap'] == overlap]
    if not sub.empty:
        axes[0].plot(sub['chunk_size'], sub['num_chunks'],
                     marker='o', label=f'overlap={overlap}')
axes[0].set_title('Number of Chunks vs Chunk Size')
axes[0].set_xlabel('Chunk Size (chars)')
axes[0].set_ylabel('Number of Chunks')
axes[0].legend()

# Plot 2: Avg token count per chunk
for overlap in OVERLAPS:
    sub = df[df['overlap'] == overlap]
    if not sub.empty:
        axes[1].plot(sub['chunk_size'], sub['avg_tokens'],
                     marker='s', label=f'overlap={overlap}')
axes[1].axhline(y=512, color='red', linestyle='--', alpha=0.6, label='GPT-4o context safe limit')
axes[1].set_title('Avg Token Count per Chunk')
axes[1].set_xlabel('Chunk Size (chars)')
axes[1].set_ylabel('Avg Tokens')
axes[1].legend()

plt.tight_layout()
plt.savefig('chunking_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
INSPECT_SIZE    = 512
INSPECT_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=INSPECT_SIZE,
    chunk_overlap=INSPECT_OVERLAP,
    separators=['\n\n', '\n', '.', ' ', '']
)
chunks = splitter.split_text(full_text)

print(f'Config: size={INSPECT_SIZE}, overlap={INSPECT_OVERLAP}')
print(f'Total chunks: {len(chunks)}\n')

for i, chunk in enumerate(chunks[:5]):
    tokens = count_tokens(chunk)
    print(f'{'─'*60}')
    print(f'Chunk {i+1} | chars={len(chunk)} | tokens={tokens}')
    print(chunk)
    print()

In [ ]:
token_counts = [count_tokens(c) for c in chunks]

plt.figure(figsize=(10, 4))
plt.hist(token_counts, bins=30, color='#7c6af7', edgecolor='white', alpha=0.85)
plt.axvline(x=512, color='red',    linestyle='--', label='512 token mark')
plt.axvline(x=sum(token_counts)/len(token_counts),
            color='orange', linestyle='--', label=f'Mean ({sum(token_counts)//len(token_counts)} tokens)')
plt.title(f'Token Distribution (size={INSPECT_SIZE}, overlap={INSPECT_OVERLAP})')
plt.xlabel('Tokens per Chunk')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def check_overlap(chunks: list[str], expected_overlap: int):
    """Verifies that consecutive chunks share text (overlap > 0)."""
    mismatches = 0
    for i in range(len(chunks) - 1):
        end_of_prev  = chunks[i][-expected_overlap:]
        start_of_next = chunks[i+1][:expected_overlap]
        # Check if they share at least 50% of expected overlap
        shared = sum(a == b for a, b in zip(end_of_prev, start_of_next))
        if shared < expected_overlap * 0.5:
            mismatches += 1

    print(f'Chunks checked : {len(chunks) - 1}')
    print(f'Overlap issues : {mismatches}')
    print(f'Overlap health : {"✓ Good" if mismatches == 0 else "⚠ Review needed"}')

check_overlap(chunks, INSPECT_OVERLAP)

In [ ]:
# ── FILL IN after reviewing the grid search and histogram ──
RECOMMENDED_CHUNK_SIZE    = 512
RECOMMENDED_CHUNK_OVERLAP = 100

print('Recommended config to copy into backend/core/config.py:')
print(f'  CHUNK_SIZE    = {RECOMMENDED_CHUNK_SIZE}')
print(f'  CHUNK_OVERLAP = {RECOMMENDED_CHUNK_OVERLAP}')

# Double-check: no chunk exceeds 600 tokens (GPT-4o safe limit)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=RECOMMENDED_CHUNK_SIZE,
    chunk_overlap=RECOMMENDED_CHUNK_OVERLAP
)
final_chunks = splitter.split_text(full_text)
oversized = [c for c in final_chunks if count_tokens(c) > 600]
print(f'\nOversized chunks (>600 tokens): {len(oversized)}')
if oversized:
    print('  ⚠  Consider reducing chunk_size.')
else:
    print('  ✓  All chunks within safe token limit.')